# 👕 **Fashion MNIST Classification with PyTorch CNN**
# **Master Convolutional Networks in PyTorch**

---

## 🎓 **Learning Objectives**

By the end of this lesson, you will be able to:

✅ Build **CNNs in PyTorch** - Conv2d, MaxPool2d layers
✅ Understand **PyTorch CNN architecture** - Different from Keras
✅ Classify **Fashion MNIST** - 10 clothing categories
✅ Achieve **88%+ accuracy** with simple CNN
✅ Compare **PyTorch CNN vs TensorFlow CNN**
✅ Master **PyTorch computer vision** patterns

---

## 📚 **Prerequisites**

- ✅ PyTorch fundamentals (previous MNIST lesson)
- ✅ CNN concepts (Lesson 08)
- ✅ Image preprocessing

**Estimated Time:** 1-2 hours

---

## 🌟 **Why Fashion MNIST?**

### **More Challenging Than MNIST Digits:**

**MNIST Digits (Easy):**
```
Accuracy: 98-99% (almost perfect!)
Classes: 10 digits (0-9)
Patterns: Simple curves and lines
Challenge: Too easy for modern deep learning
```

**Fashion MNIST (Medium):**
```
Accuracy: 88-92% (more realistic!)
Classes: 10 clothing items
Patterns: Complex textures, shapes, orientations
Challenge: Perfect for learning CNNs! ✨
```

### **Fashion MNIST Dataset:**

- **Images:** 70,000 grayscale images (28×28)
- **Training:** 60,000 images
- **Testing:** 10,000 images
- **Classes:** 10 clothing categories

**Categories:**
```
0: T-shirt/top       👕
1: Trouser           👖
2: Pullover          🧥
3: Dress             👗
4: Coat              🧥
5: Sandal            👡
6: Shirt             👔
7: Sneaker           👟
8: Bag               👜
9: Ankle boot        👢
```

---

## 🏗️ **PyTorch CNN Architecture**

### **Our CNN Model:**

```python
class FashionMNISTCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Fully connected layers
        self.fc_hidden = nn.Linear(13*13*32, 100)
        self.fc_output = nn.Linear(100, 10)

    def forward(self, x):
        # Conv + ReLU + Pool
        x = self.conv1(x)        # (batch, 1, 28, 28) → (batch, 32, 26, 26)
        x = F.relu(x)
        x = self.pool(x)         # (batch, 32, 26, 26) → (batch, 32, 13, 13)

        # Flatten
        x = x.view(-1, 13*13*32) # (batch, 32, 13, 13) → (batch, 5408)

        # Fully connected
        x = self.fc_hidden(x)    # (batch, 5408) → (batch, 100)
        x = F.relu(x)
        x = self.fc_output(x)    # (batch, 100) → (batch, 10)

        return x
```

### **Layer-by-Layer Breakdown:**

**1. Conv2d Layer:**
```python
nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, stride=1)

Input:  (batch, 1, 28, 28)   # 1 grayscale channel
Kernel: 3×3 filters × 32      # 32 different filters
Output: (batch, 32, 26, 26)   # 32 feature maps

Output size: (28 - 3) + 1 = 26
```

**2. MaxPool2d Layer:**
```python
nn.MaxPool2d(kernel_size=2, stride=2)

Input:  (batch, 32, 26, 26)
Pool:   2×2 window, stride 2  # Take max in each 2×2 block
Output: (batch, 32, 13, 13)   # Halved spatial dimensions

Output size: 26 / 2 = 13
```

**3. Flatten:**
```python
x.view(-1, 13*13*32)

Input:  (batch, 32, 13, 13)
Output: (batch, 5408)         # 32 × 13 × 13 = 5408

Purpose: Convert 3D feature maps to 1D vector for FC layers
```

**4. Fully Connected Layers:**
```python
nn.Linear(5408, 100)  # 5408 → 100 (feature extraction)
nn.Linear(100, 10)    # 100 → 10 (classification)
```

---

## 🔬 **PyTorch CNN vs TensorFlow CNN**

### **Side-by-Side Comparison:**

**TensorFlow/Keras:**
```python
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])
```

**PyTorch:**
```python
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(13*13*32, 100)
        self.fc2 = nn.Linear(100, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(-1, 13*13*32)
        x = F.relu(self.fc1(x))
        return self.fc2(x)
```

**Key Differences:**

| Aspect | TensorFlow/Keras | PyTorch |
|--------|------------------|---------|
| **Layer definition** | In Sequential list | In __init__ |
| **Activation** | As parameter | In forward() |
| **Flatten** | Separate layer | x.view() |
| **Input shape** | Specified explicitly | Inferred |
| **Channel order** | (H, W, C) | **(C, H, W)** ⚠️ |

**CRITICAL: PyTorch uses (Channel, Height, Width) ordering!**

---

Let's build a Fashion MNIST CNN classifier! 👕🚀

## Steps to be followed:

1. Import necessary libraries
2. Data Preprocessing Setup
3. Load Datasets
4. Initialize DataLoaders
5. Define the CNN Model
6. Set Up Loss Function and Optimizer
7. Training the Model
8. Evaluate the Model


### Step 1: Import necessary libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
import torch.nn.functional as F

### Step 2:  Data Preprocessing Setup
- Define transformations for the input data:
     - `ToTensor():` Converts image data from PIL format or NumPy arrays to PyTorch tensors.
     - `Normalize((0.5,), (0.5,)):` Normalizes tensor images using mean = 0.5 and std = 0.5.

In [2]:
# Data preprocessing: normalization
transform_nm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

### Step 3: Load Datasets
- **Train Dataset:** Load FashionMNIST training data, applying the defined transformations.
- **Test Dataset:** Load FashionMNIST test data similarly.

In [3]:
# Loading the dataset
train_dataset_nm = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform_nm)
test_dataset_nm = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform_nm)

100%|██████████| 26421880/26421880 [00:01<00:00, 18498656.01it/s]


Extracting ./data/FashionMNIST/raw/train-images-idx3-ubyte.gz to ./data/FashionMNIST/raw



100%|██████████| 29515/29515 [00:00<00:00, 338885.53it/s]


Extracting ./data/FashionMNIST/raw/train-labels-idx1-ubyte.gz to ./data/FashionMNIST/raw



100%|██████████| 4422102/4422102 [00:00<00:00, 6090756.96it/s]


Extracting ./data/FashionMNIST/raw/t10k-images-idx3-ubyte.gz to ./data/FashionMNIST/raw



100%|██████████| 5148/5148 [00:00<00:00, 3774213.77it/s]

Extracting ./data/FashionMNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/FashionMNIST/raw



### Step 4: Initialize DataLoaders
- `Training DataLoader:` Batches, shuffles, and prepares training data for processing in the model.
- `Testing DataLoader:` Batches and prepares test data for evaluation (shuffling is not necessary for testing)

In [4]:
train_loader_nm = torch.utils.data.DataLoader(train_dataset_nm, batch_size=32, shuffle=True)
test_loader_nm = torch.utils.data.DataLoader(test_dataset_nm, batch_size=32, shuffle=False)

# Printing the shape of the datasets
print(f'Training data: {len(train_dataset_nm)} samples')
print(f'Testing data: {len(test_dataset_nm)} samples')

Training data: 60000 samples
Testing data: 10000 samples


### Step 5: Define the CNN Model
- **Model Class Initialization:** Set up the neural network structure with layers defined in the `__init__` method.
- **Layers:** Include one convolutional layer, one pooling layer, and two fully connected layers.
- **Data Flow through Layers:** Define how data moves through the model using the forward method, incorporating activations (ReLU) and pooling

In [5]:
# Define a convolutional neural network for classifying FashionMNIST dataset images
class FashionMNISTCNN(nn.Module):
    def __init__(self):
        super(FashionMNISTCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc_hidden = nn.Linear(13*13*32, 100)
        self.fc_output = nn.Linear(100, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        x = x.view(-1, 13*13*32)
        x = self.fc_hidden(x)
        x = F.relu(x)
        x = self.fc_output(x)
        return x



### Step 6: Set Up Loss Function and Optimizer

- **CrossEntropyLoss:** Used for multi-class classification tasks.
- **Adam Optimizer:** A method for stochastic optimization with a set learning rate of 0.001.

In [6]:
model = FashionMNISTCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### Step 7: Training the Model and saving it at each epoch

- **Epoch Iteration:** Loop through the dataset multiple times (epochs).
- **Batch Processing:** For each batch in the DataLoader, perform forward pass, loss calculation, backpropagation, and parameter update.
- **Save Model State:** Save the model after training to reuse it later without needing to retrain.

In [7]:
# Training the model for 10 epochs
# Define the path for saving the model
model_path = './mnist_model.pth'

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images_nm, labels_nm in train_loader_nm:
        optimizer.zero_grad()

        outputs_nm = model(images_nm)
        loss = criterion(outputs_nm, labels_nm)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted_train = torch.max(outputs_nm.data, 1)
        total_train += labels_nm.size(0)
        correct_train += (predicted_train == labels_nm).sum().item()

    train_accuracy = 100 * correct_train / total_train
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader_nm)}, Train Accuracy: {train_accuracy}%")

    # Save the model at the end of each epoch
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to {model_path}")

Epoch 1/10, Loss: 0.39879183058540024, Train Accuracy: 85.74666666666667%
Model saved to ./mnist_model.pth
Epoch 2/10, Loss: 0.26612817190289495, Train Accuracy: 90.37333333333333%
Model saved to ./mnist_model.pth
Epoch 3/10, Loss: 0.2216284805228313, Train Accuracy: 91.945%
Model saved to ./mnist_model.pth
Epoch 4/10, Loss: 0.1883942069063584, Train Accuracy: 93.05%
Model saved to ./mnist_model.pth
Epoch 5/10, Loss: 0.1620183053433895, Train Accuracy: 94.01166666666667%
Model saved to ./mnist_model.pth
Epoch 6/10, Loss: 0.13680088447729746, Train Accuracy: 94.945%
Model saved to ./mnist_model.pth
Epoch 7/10, Loss: 0.1175761468090117, Train Accuracy: 95.71%
Model saved to ./mnist_model.pth
Epoch 8/10, Loss: 0.09986156567347547, Train Accuracy: 96.26666666666667%
Model saved to ./mnist_model.pth
Epoch 9/10, Loss: 0.08623724563966195, Train Accuracy: 96.845%
Model saved to ./mnist_model.pth
Epoch 10/10, Loss: 0.074453572653234, Train Accuracy: 97.26166666666667%
Model saved to ./mnist_mo

### Step 8:  Evaluate the Model
- **Switch to Evaluation Mode:** Ensure the model is in eval mode to disable dropout or batch norm effects during testing.
- **Accuracy Calculation:** Compare the model’s output to the true labels, calculate overall accuracy.

In [8]:
# Test the model
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images_nm, labels_nm in test_loader_nm:
        outputs_nm = model(images_nm)
        _, predicted = torch.max(outputs_nm.data, 1)
        total += labels_nm.size(0)
        correct += (predicted == labels_nm).sum().item()

accuracy = 100 * correct / total
print(f"Model accuracy on test set: {accuracy}%")

Model accuracy on test set: 91.45%


---

# 🎯 **Lesson Summary: PyTorch CNN for Fashion MNIST**

---

## 🏆 **Key Takeaways**

### **1. PyTorch CNNs Are Powerful Yet Explicit**

✅ **Conv2d layers** - Learn spatial features (edges, textures)
✅ **MaxPool2d** - Reduce spatial dimensions, computational cost
✅ **Explicit forward()** - Full control over data flow
✅ **88%+ accuracy** on Fashion MNIST (harder than digits!)
✅ **Channel-first ordering** - (C, H, W) not (H, W, C)

### **2. Complete PyTorch Workflow Mastered**

```
Data Loading → Transforms → DataLoader → Model → Training Loop → Evaluation
All explicit, all controllable! ✨
```

---

## 📊 **Performance Analysis**

**Expected Results:**
```
Epoch 1:  Loss: 0.52, Train Acc: 82%
Epoch 5:  Loss: 0.32, Train Acc: 88%
Epoch 10: Loss: 0.25, Train Acc: 91%

Test Accuracy: 88-90% ✨
```

**Performance by Class:**
```
Easy (90%+):  Trouser 👖, Sandal 👡, Sneaker 👟, Bag 👜
Medium (85%): T-shirt 👕, Pullover 🧥, Ankle boot 👢
Hard (75%):   Shirt 👔, Coat 🧥, Dress 👗

Why hard? Similar patterns (Shirt vs T-shirt, Coat vs Pullover)
```

---

## 🔧 **PyTorch CNN Patterns**

### **Standard CNN Template:**

```python
class MyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolutional blocks
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size)
        self.pool = nn.MaxPool2d(2, 2)

        # Calculate flattened size: (H//2) * (W//2) * out_channels
        self.fc1 = nn.Linear(flattened_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)  # Flatten
        x = F.relu(self.fc1(x))
        return self.fc2(x)
```

---

## 💡 **Improvements to Try**

### **1. Deeper CNN:**
```python
self.conv1 = nn.Conv2d(1, 32, 3)
self.conv2 = nn.Conv2d(32, 64, 3)  # Add second conv layer!
self.pool = nn.MaxPool2d(2, 2)

# Forward:
x = self.pool(F.relu(self.conv1(x)))
x = self.pool(F.relu(self.conv2(x)))

Expected: +2-3% accuracy
```

### **2. Batch Normalization:**
```python
self.conv1 = nn.Conv2d(1, 32, 3)
self.bn1 = nn.BatchNorm2d(32)  # Normalize activations

# Forward:
x = F.relu(self.bn1(self.conv1(x)))

Expected: Faster training, +1-2% accuracy
```

### **3. Dropout:**
```python
self.dropout = nn.Dropout(0.5)

# Forward (after flatten):
x = self.dropout(F.relu(self.fc1(x)))

Expected: Better generalization
```

---

## 🌟 **Congratulations!**

You've mastered:

✅ **PyTorch CNNs** - Conv2d, MaxPool2d, Flatten
✅ **Fashion MNIST** - 88%+ accuracy on clothing classification
✅ **PyTorch fundamentals** - Both FC networks and CNNs
✅ **Framework diversity** - Can work in PyTorch AND TensorFlow!

**You can now:**
- Build CNNs in PyTorch
- Train computer vision models
- Work in research (PyTorch) or production (TensorFlow)
- Read and implement research papers
- **Compete with industry professionals!**

**Lesson 06 PyTorch COMPLETE!** 🔥🚀

---

## 📖 **Additional Resources**

**PyTorch Vision:**
- TorchVision Models: https://pytorch.org/vision/stable/models.html
- Pre-trained CNNs: ResNet, VGG, EfficientNet
- Datasets: CIFAR, ImageNet, COCO

**Advanced Topics:**
- Transfer Learning in PyTorch
- Custom loss functions
- Multi-GPU training
- PyTorch Lightning (high-level API)

---

**You're now a PyTorch expert! Both fundamentals and CNNs mastered! 👕🔥🚀**